# ML-05 — Feature Vector and Leakage/Privacy Check

[![Open In Colab](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/jerovernay/FlyRank-Internship/blob/main/work/notebooks/w03_feature_leakage_check.ipynb?flush_cache=true)

This skeleton is yours to fill. Work the sections **in order** — each one has a one-line hint. Simple words, honest numbers.

> Working with an AI assistant? Tell it to read `skills/README.md` first and load the one skill this assignment names on its card.

## 1. Build the feature vector

Same grain as the data contract (ML-04): one row = one content item, aggregated over
`month=2026-03` from `fact_content_daily_performance`, joined to `dim_content` for
categorical/context columns. This time the goal is the actual, ready-to-model matrix,
not just the field list — so every feature gets an explicit missing-value rule and
categoricals are encoded.

Engineered on top of the 5 core features from ML-04:
- `engagement_rate = engaged_sessions / total_impressions` (guarded against divide-by-zero)
- `has_word_count` flag, because `word_count` missingness follows `content_type`
  (per the `flyrank-data` skill gotcha) — a blind `fillna(0)` would inject a fake
  "short content" signal into whichever type is missing it most
- `content_type` and `main_intent` one-hot encoded (clustering needs numeric input)

In [1]:
import duckdb
import pandas as pd

# HF_TOKEN from local .env — never pasted into a cell (public repo)
with open("../../.env") as f:
    for line in f:
        if line.startswith("HF_TOKEN="):
            os_token = line.strip().split("=", 1)[1]

import os
os.environ["HF_TOKEN"] = os_token

con = duckdb.connect()
con.sql(f"CREATE OR REPLACE SECRET hf (TYPE huggingface, TOKEN '{os_token}')")

MONTH = "hf://datasets/FlyRank/internship-warehouse/fact_content_daily_performance/month=2026-03/*.parquet"
DIM_CONTENT = "hf://datasets/FlyRank/internship-warehouse/dim_content.parquet"

raw = con.sql(f"""
    WITH gsc_agg AS (
        SELECT content_hash_id,
               SUM(gsc_impressions) AS total_impressions,
               SUM(gsc_clicks) AS total_clicks,
               SUM(gsc_impressions * gsc_avg_position) / NULLIF(SUM(gsc_impressions), 0) AS avg_position
        FROM read_parquet('{MONTH}') GROUP BY content_hash_id
    ),
    ga4_agg AS (
        SELECT content_hash_id, SUM(ga4_engaged_sessions) AS engaged_sessions
        FROM read_parquet('{MONTH}') WHERE ga4_data_available IS TRUE GROUP BY content_hash_id
    )
    SELECT g.content_hash_id, g.total_impressions, g.total_clicks, g.avg_position,
           COALESCE(a.engaged_sessions, 0) AS engaged_sessions,
           dc.word_count, dc.content_type, dc.main_intent
    FROM gsc_agg g
    LEFT JOIN ga4_agg a USING (content_hash_id)
    LEFT JOIN read_parquet('{DIM_CONTENT}') dc USING (content_hash_id)
""").df()

feature_df = raw.copy()

# engagement_rate: guard divide-by-zero (total_impressions can be 0)
feature_df["engagement_rate"] = (
    feature_df["engaged_sessions"] / feature_df["total_impressions"].replace(0, pd.NA)
).fillna(0)

# word_count: flag missingness instead of blind fillna (missingness follows content_type)
feature_df["has_word_count"] = feature_df["word_count"].notna().astype(int)
feature_df["word_count"] = feature_df["word_count"].fillna(0)

# categoricals -> one-hot
feature_df = pd.get_dummies(feature_df, columns=["content_type", "main_intent"], dummy_na=True)

print("rows:", len(feature_df), "| cols:", feature_df.shape[1])
print("duplicate content_hash_id:", feature_df["content_hash_id"].duplicated().sum())
feature_df.head(3)


rows: 331437 | cols: 17
duplicate content_hash_id: 0


C:\Users\jeron\AppData\Local\Temp\ipykernel_11560\3194683125.py:44: FutureWarning: Downcasting object dtype arrays on .fillna, .ffill, .bfill is deprecated and will change in a future version. Call result.infer_objects(copy=False) instead. To opt-in to the future behavior, set `pd.set_option('future.no_silent_downcasting', True)`
  ).fillna(0)


,content_hash_id,total_impressions,total_clicks,avg_position,engaged_sessions,word_count,engagement_rate,has_word_count,content_type_comparison article,content_type_feedly article,content_type_keyword article,content_type_nan,main_intent_commercial,main_intent_informational,main_intent_navigational,main_intent_transactional,main_intent_nan
0,content_956bca9e7b3fe3c6,14.0,0.0,5.928571,0.0,3589,0.0,1,False,False,True,False,False,True,False,False,False
1,content_8f0fb33cd4ab18bc,0.0,0.0,NaN,0.0,3489,0.0,1,False,False,True,False,False,True,False,False,False
2,content_bde9f552addd7a51,2842.0,9.0,5.568614,0.0,3749,0.0,1,False,False,True,False,False,True,False,False,False


## 2. Feature notes (meaning, missing, categorical, available-when?)

| Feature | Meaning | Missing | Categorical? | Available-when |
|---|---|---|---|---|
| `total_impressions` | Sum of GSC search impressions over the month | Never NULL — `SUM()` over 0 rows would be missing, but every content item has ≥1 GSC row in this partition; 0 is a real "no impressions" value | No | Known at end of month (GSC reports daily, no future data used) |
| `total_clicks` | Sum of GSC clicks over the month | Same as above — real 0s, not missing | No | Same |
| `avg_position` | Impression-weighted average GSC rank | `NaN` when `total_impressions = 0` (no impressions → no position to average) — **left as NaN, not filled**, because 0 would look like "rank #0" (the flyrank-data gotcha) | No | Same |
| `engaged_sessions` | Sum of GA4 engaged sessions, counted only where `ga4_data_available IS TRUE` | Filled with 0 via `COALESCE` — but this 0 conflates "verified zero engagement" with "no GA4 coverage at all" (content-level GA4 coverage is only ~27%, per ML-04). This is a known limitation, not a clean fix here. | No | Known at end of month, GA4-covered clients only |
| `word_count` | Content length from `dim_content` | ~30% missing, correlated with `content_type` (per flyrank-data skill) — filled with 0 **plus** a companion `has_word_count` flag so the model can tell "short" from "unknown" | No | Static content attribute, known before any traffic exists |
| `has_word_count` | 1 if `word_count` was present, 0 if it had to be filled | Never missing by construction | No (binary flag) | Same as `word_count` |
| `engagement_rate` | `engaged_sessions / total_impressions`, 0 when impressions = 0 | Never NaN — guarded divide-by-zero | No | Same as its inputs |
| `content_type_*`, `main_intent_*` | One-hot dummies from `dim_content` | Missing category gets its own `_nan` dummy column (via `dummy_na=True`) instead of being dropped or defaulted into another category | Yes | Static content attribute, known before any traffic exists |

Nothing here uses a future window or a value that depends on the outcome being predicted —
all features are either static content attributes or aggregates over the same fixed
`month=2026-03` window used for the (proxy) unit of analysis.

In [2]:
# Verify the missingness claims made above against the actual feature_df
n = len(feature_df)
print("avg_position NaN:", feature_df["avg_position"].isna().sum(),
      f"({feature_df['avg_position'].isna().mean():.1%})")
print("has_word_count == 0 (i.e. word_count was filled):", (feature_df["has_word_count"] == 0).sum(),
      f"({(feature_df['has_word_count'] == 0).mean():.1%})")
print("content_type_nan == True:", feature_df.get("content_type_nan", pd.Series(dtype=bool)).sum())
print("main_intent_nan == True:", feature_df.get("main_intent_nan", pd.Series(dtype=bool)).sum())
print("engaged_sessions == 0 (verified zero OR no GA4 coverage):", (feature_df["engaged_sessions"] == 0).sum(),
      f"({(feature_df['engaged_sessions'] == 0).mean():.1%})")


avg_position NaN: 154699 (46.7%)
has_word_count == 0 (i.e. word_count was filled): 107429 (32.4%)
content_type_nan == True: 0
main_intent_nan == True: 58585
engaged_sessions == 0 (verified zero OR no GA4 coverage): 317626 (95.8%)


## 3. The leakage hunt

Lane 3 has no real supervised label (it's unsupervised clustering), so — same as ML-04 —
this is a demonstration leakage trap: build a proxy classification task and prove the effect
is real and measurable, not hand-waved.

**Proxy label**: `is_high_engagement` = `engaged_sessions > 0` (any measured engagement at all).
`engaged_sessions` is 95.8% zero (Section 2), so a top-20%-quantile split would have landed on
0 itself and produced a single-class label — presence/absence is the split that actually works
on this zero-inflated column.

**The leak**: `engagement_rate = engaged_sessions / total_impressions` is *engineered directly
from the label source*. It doesn't look like a duplicate column (it's a ratio, not a copy),
which is exactly what makes this kind of leak easy to miss — a well-meaning "let's add a rate
feature" step can quietly re-inject the label.

- Honest features: `total_impressions`, `total_clicks`, `word_count`, `has_word_count`
  (all independent of `engaged_sessions`)
- Leaky features: honest set + `engagement_rate`

If the leaky model scores higher than the honest one, that's the leak made visible.

**Result note**: the gap is real (0.856 → 0.877 ROC AUC) but modest, not the near-perfect
leak from ML-04's example. That's because `engagement_rate` is zero-filled whenever
`total_impressions = 0` (our own divide-by-zero guard from Section 1) — so a page can have
`engaged_sessions > 0` (label = 1) while `engagement_rate` reads 0, which breaks the leak's
correlation with the label in those rows. The lesson: a leak doesn't have to be perfect to be
a leak — a partial, engineered-feature leak like this is easier to miss than an obvious
duplicate column precisely because the score bump looks like "the feature just helps."

In [3]:
from sklearn.linear_model import LogisticRegression
from sklearn.model_selection import cross_val_score

demo = feature_df.dropna(subset=["total_impressions", "total_clicks", "word_count"]).copy()
# engaged_sessions is 95.8% zero (zero-inflated, per Section 2) so a quantile split lands
# on 0 itself — use presence/absence of any engagement instead.
demo["is_high_engagement"] = (demo["engaged_sessions"] > 0).astype(int)
print("class balance:", demo["is_high_engagement"].value_counts(normalize=True).to_dict())

honest_features = ["total_impressions", "total_clicks", "word_count", "has_word_count"]
leaky_features = honest_features + ["engagement_rate"]

# accuracy is dominated by the 95.8%/4.2% imbalance (both scores tie near 0.96 either way) —
# use roc_auc so the leak isn't masked by predicting the majority class
honest_score = cross_val_score(
    LogisticRegression(max_iter=1000), demo[honest_features], demo["is_high_engagement"],
    cv=3, scoring="roc_auc"
).mean()
leaky_score = cross_val_score(
    LogisticRegression(max_iter=1000), demo[leaky_features], demo["is_high_engagement"],
    cv=3, scoring="roc_auc"
).mean()

print(f"honest_score (no engagement_rate), roc_auc: {honest_score:.3f}")
print(f"leaky_score  (with engagement_rate), roc_auc: {leaky_score:.3f}")


class balance: {0: 0.9583299390231025, 1: 0.04167006097689757}


honest_score (no engagement_rate), roc_auc: 0.856
leaky_score  (with engagement_rate), roc_auc: 0.877


In [4]:
# Test the intuition: is avg_position a leak, or just a legitimate correlated feature?
demo_pos = demo.dropna(subset=["avg_position"]).copy()

corr = demo_pos["avg_position"].corr(demo_pos["is_high_engagement"])
print(f"corr(avg_position, is_high_engagement): {corr:.3f}")

honest_score_pos = cross_val_score(
    LogisticRegression(max_iter=1000), demo_pos[honest_features], demo_pos["is_high_engagement"],
    cv=3, scoring="roc_auc"
).mean()
with_position_score = cross_val_score(
    LogisticRegression(max_iter=1000), demo_pos[honest_features + ["avg_position"]], demo_pos["is_high_engagement"],
    cv=3, scoring="roc_auc"
).mean()

print(f"honest_score (no avg_position), roc_auc: {honest_score_pos:.3f}")
print(f"with avg_position, roc_auc: {with_position_score:.3f}")


corr(avg_position, is_high_engagement): -0.029


honest_score (no avg_position), roc_auc: 0.827
with avg_position, roc_auc: 0.824


## 4. What I excluded and why

| Excluded field | Why |
|---|---|
| `content_hash_id`, `client_hash_id`, `keyword_hash_id`, `url_hash_id` | Identifiers, not signal — used only for joins/grouping (ML-04) |
| `engagement_rate` | Proven leak in Section 3 (ROC AUC 0.856 → 0.877 when included) — engineered directly from the label source `engaged_sessions` |
| `ga4_*` columns where `ga4_data_available = FALSE` | Zero-filled placeholders, not real zeros — including them would teach the model a fake "zero engagement" signal for clients whose GA4 just isn't reporting (ML-04 finding) |
| Raw GA4 columns outside the 5-feature frame (`ga4_sessions`, `scroll_events`, etc.) | Same reliability problem as above — only ~27% of content has any real GA4 row in this month, so any of these would carry the same coverage bias |

**Tested and kept, not excluded**: `avg_position` was suspected of the same "rank drives clicks
drives engagement" leak as `engagement_rate`, but the test says otherwise — `corr(avg_position,
is_high_engagement) = -0.029`, and adding it to the honest features barely moves the score
(0.827 → 0.824, within cross-val noise). It stays in the feature set; the intuition was
reasonable to check, but the data didn't back it up.

In [5]:
# Confirm the excluded fields never made it into the final modeling feature list
# (engagement_rate stays IN feature_df as an audit trail of the leak test — it's excluded
# from honest_features, not deleted from the dataframe)
excluded = ["content_hash_id", "client_hash_id", "keyword_hash_id", "url_hash_id",
            "engagement_rate", "ga4_sessions", "scroll_events"]
leaked_in = [c for c in excluded if c in honest_features]
print("excluded fields present in the modeling feature list (should be empty):", leaked_in)
print("final feature columns used for modeling:", honest_features)


excluded fields present in the modeling feature list (should be empty): []
final feature columns used for modeling: ['total_impressions', 'total_clicks', 'word_count', 'has_word_count']


## Self-check

Before you submit, confirm each line honestly:

- [x] Every section above is filled — markdown thinking AND the code that backs it
- [x] The notebook runs top to bottom with no errors (Runtime → Run all)
- [x] No client names, URLs, or private queries anywhere
- [x] My claims use careful words: observed, measured, directional, decision-support
- [ ] Committed to my repo under `work/notebooks/` — then submit your repo URL on the card. Done.